# Cleaning & Normalization

Cleaning is the process of making extracted document content more consistent **without changing its meaning**.

This notebook uses the documents in `data/`.

The workflow is:

```text
Real document
      ↓
Actual parser
      ↓
Actual extracted output
      ↓
Inspect
      ↓
Identify an actual issue
      ↓
Clean it if justified
      ↓
Compare before / after
      ↓
Validate
```

If a particular artifact is not present, we will record that fact and move on.


## Learning Objectives

By the end of this notebook, you should be able to:

- inspect real parser output before cleaning;
- distinguish normalization from semantic correction;
- normalize whitespace when the actual output contains redundant whitespace;
- normalize Unicode representation;
- clean excessive blank-line repetition when it actually occurs;
- use source structure when it provides a better cleaning signal;
- avoid destroying table, heading, or reading-order information;
- validate that useful document content survived cleaning.


## 1. The Core Principle

A document-ingestion engineer should never begin by assuming that a document has a particular extraction problem.

Instead:

```text
Inspect → Identify → Transform → Verify
```

For example, if:

```python
raw_text.count("\f")
```

returns `0`, that means the actual parser output contains no form-feed characters.

A zero is useful information.


## 2. Install the Libraries


In [1]:
!pip install -q pymupdf python-docx beautifulsoup4


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 3. Imports


In [2]:
from pathlib import Path
import re
import unicodedata

import pymupdf
from docx import Document
from bs4 import BeautifulSoup

# Part A — PDF

## 4. Extract the Actual PDF



In [3]:
DATA_DIR = Path("data")
PDF_PATH = DATA_DIR / "employee_travel_policy.pdf"

with pymupdf.open(PDF_PATH) as pdf:
    pdf_pages = [page.get_text("text") for page in pdf]

pdf_raw = "\n".join(pdf_pages)

print(pdf_raw)

Employee Travel and Reimbursement Policy
Document ID: HR-TRV-2026-003 | Version: 4.0 | Effective: 1 July 2026
1. Purpose
This policy establishes the requirements for business travel, eligible expenses, receipts, approvals,
and reimbursement. It applies to employees travelling on approved company business.
2. Travel Approval
Employees must obtain manager approval before booking travel. International travel also requires
approval from the relevant department head.
3. Accommodation Limits
Location
Nightly limit
Notes
Lagos
₦120,000
Standard business accommodation
Abuja
₦110,000
Standard business accommodation
Port Harcourt
₦100,000
Standard business accommodation
International
Actual reasonable cost
Department approval required
4. Transportation
Employees should use reasonable and cost-effective transportation. Air travel should normally be
booked in economy class unless an approved exception applies.
5. Receipts and Reimbursement
Expense claims should be submitted within 10 business days

## 5. Inspect the PDF Representation

In [4]:
print(repr(pdf_raw))

print("\nCharacter profile")
print("LF:", pdf_raw.count("\n"))
print("CRLF:", pdf_raw.count("\r\n"))
print("CR:", pdf_raw.count("\r"))
print("Tabs:", pdf_raw.count("\t"))
print("Form feeds:", pdf_raw.count("\f"))
print("Repeated spaces:", len(re.findall(r" {2,}", pdf_raw)))

'Employee Travel and Reimbursement Policy\nDocument ID: HR-TRV-2026-003 | Version: 4.0 | Effective: 1 July 2026\n1. Purpose\nThis policy establishes the requirements for business travel, eligible expenses, receipts, approvals,\nand reimbursement. It applies to employees travelling on approved company business.\n2. Travel Approval\nEmployees must obtain manager approval before booking travel. International travel also requires\napproval from the relevant department head.\n3. Accommodation Limits\nLocation\nNightly limit\nNotes\nLagos\n₦120,000\nStandard business accommodation\nAbuja\n₦110,000\nStandard business accommodation\nPort Harcourt\n₦100,000\nStandard business accommodation\nInternational\nActual reasonable cost\nDepartment approval required\n4. Transportation\nEmployees should use reasonable and cost-effective transportation. Air travel should normally be\nbooked in economy class unless an approved exception applies.\n5. Receipts and Reimbursement\nExpense claims should be subm

### What this tells us

These values describe the actual parser output.

For this PDF, do not infer a problem from a generic list of possible PDF artifacts. If the count is zero, the artifact is not present in this extraction.

For example, if `Form feeds` is `0`, there is no form-feed cleanup to demonstrate here.


## 6. Line-Ending Normalization

We can define a general normalization function, then test it against the real PDF output.


In [5]:
def normalize_line_endings(text):
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    return text

pdf_line_normalized = normalize_line_endings(pdf_raw)

print("Changed:", pdf_line_normalized != pdf_raw)

Changed: False


If `Changed` is `False`, the PDF already uses the representation we want.

That is a successful inspection result, not a failed demonstration.


## 7. Unicode Normalization

Apply NFC normalization to the actual PDF extraction.


In [6]:
pdf_unicode_normalized = unicodedata.normalize(
    "NFC",
    pdf_line_normalized
)

print("Changed:", pdf_unicode_normalized != pdf_line_normalized)

Changed: False


Again, a `False` result simply means the actual extracted text did not require this normalization.


## 8. Horizontal Whitespace

Check the actual PDF for repeated spaces or tabs.


In [7]:
pdf_horizontal_issues = [
    line
    for line in pdf_raw.splitlines()
    if re.search(r"[ \t]{2,}", line)
]

print("Lines with repeated horizontal whitespace:",
      len(pdf_horizontal_issues))

for line in pdf_horizontal_issues:
    print(repr(line))

Lines with repeated horizontal whitespace: 0


If no lines are returned, we do not claim that this PDF demonstrates a repeated-whitespace problem.

The PDF is still useful: it shows why profiling the real extraction matters.


## 9. Preserve the PDF Content

Inspect the final representation before passing it to the next ingestion stage.


In [8]:
pdf_clean = pdf_unicode_normalized.strip()

for number, line in enumerate(pdf_clean.splitlines(), start=1):
    print(f"{number:02d}: {line}")

01: Employee Travel and Reimbursement Policy
02: Document ID: HR-TRV-2026-003 | Version: 4.0 | Effective: 1 July 2026
03: 1. Purpose
04: This policy establishes the requirements for business travel, eligible expenses, receipts, approvals,
05: and reimbursement. It applies to employees travelling on approved company business.
06: 2. Travel Approval
07: Employees must obtain manager approval before booking travel. International travel also requires
08: approval from the relevant department head.
09: 3. Accommodation Limits
10: Location
11: Nightly limit
12: Notes
13: Lagos
14: ₦120,000
15: Standard business accommodation
16: Abuja
17: ₦110,000
18: Standard business accommodation
19: Port Harcourt
20: ₦100,000
21: Standard business accommodation
22: International
23: Actual reasonable cost
24: Department approval required
25: 4. Transportation
26: Employees should use reasonable and cost-effective transportation. Air travel should normally be
27: booked in economy class unless an approved

Notice that cleaning has not attempted to "repair" uncertain content.

Cleaning is not the place to guess:

- table column boundaries;
- missing characters;
- currency values;
- names;
- identifiers;
- semantic corrections.

Those require additional evidence and belong to later processing stages.


# Part B — DOCX

## 10. Extract the DOCX

In [9]:
DOCX_PATH = DATA_DIR / "q3_customer_support_operating_brief.docx"

document = Document(DOCX_PATH)

docx_paragraphs = [
    paragraph.text
    for paragraph in document.paragraphs
]

for number, paragraph in enumerate(docx_paragraphs, start=1):
    print(f"{number:02d}: {paragraph!r}")


01: ''
02: 'Q3 Customer Support Operating Brief'
03: 'Document ID: OPS-SUP-2026-Q3    Version: 1.0'
04: 'Prepared by Customer Operations | 22 August 2026'
05: 'Executive Summary'
06: 'The support team is prioritizing faster first responses, consistent handling of refund requests, and improved escalation for cases that require specialist review.'
07: 'Service Targets'
08: 'The following targets apply to standard customer-support requests during the quarter.'
09: 'Refund Handling'
10: 'Refund requests should include the order reference, purchase date, and reason for the request. Agents should verify eligibility before escalating a case.'
11: 'Escalation Notes'
12: 'Cases involving suspected payment fraud, account compromise, or conflicting customer records should be escalated to the appropriate specialist team.'
13: 'Document Control'
14: 'Owner: Customer Operations\nReview frequency: Quarterly\nNext scheduled review: 30 November 2026'
15: ''
16: ''


Now we inspect the actual DOCX output.

The document contains a real example of multiple spaces in the `Document ID` line:

```text
Document ID: OPS-SUP-2026-Q3    Version: 1.0
```

This is an actual extracted paragraph, so we can demonstrate whitespace normalization on it.


## 11. Locate the Actual Whitespace



In [10]:
whitespace_lines = [
    paragraph
    for paragraph in docx_paragraphs
    if re.search(r"[ \t]{2,}", paragraph)
]

print("Found:", len(whitespace_lines))

for line in whitespace_lines:
    print(repr(line))


Found: 1
'Document ID: OPS-SUP-2026-Q3    Version: 1.0'


We have now observed the artifact before changing it.


## 12. Normalize the Actual Whitespace



In [11]:
def normalize_horizontal_whitespace(text):
    return re.sub(r"[ \t]+", " ", text)

docx_raw = "\n".join(docx_paragraphs)
docx_clean = normalize_horizontal_whitespace(docx_raw)

before = [
    line
    for line in docx_raw.splitlines()
    if "Document ID:" in line
][0]

after = [
    line
    for line in docx_clean.splitlines()
    if "Document ID:" in line
][0]

print("BEFORE:")
print(repr(before))

print("\nAFTER:")
print(repr(after))


BEFORE:
'Document ID: OPS-SUP-2026-Q3    Version: 1.0'

AFTER:
'Document ID: OPS-SUP-2026-Q3 Version: 1.0'


This is the complete demonstration:

```text
actual DOCX
    ↓
actual parser output
    ↓
actual repeated whitespace
    ↓
normalization
    ↓
observed before / after
```

The important point is that the input was not invented.


## 13. Check That the Normalization Did Not Remove Content



In [12]:
important_docx_values = [
    "OPS-SUP-2026-Q3",
    "Version: 1.0",
    "Executive Summary",
    "Refund Handling",
    "Document Control",
]

for value in important_docx_values:
    print(f"{value!r}: {value in docx_clean}")


'OPS-SUP-2026-Q3': True
'Version: 1.0': True
'Executive Summary': True
'Refund Handling': True
'Document Control': True


# Part C — HTML

## 14. Extract the HTML

In [13]:
HTML_PATH = DATA_DIR / "support_refunds.html"

html_source = HTML_PATH.read_text(encoding="utf-8")
soup = BeautifulSoup(html_source, "html.parser")

html_raw = soup.get_text("\n")

print(html_raw)









Support Knowledge Base — Refunds












Home


Billing


Support










Billing


Refunds and Cancellations


Updated 18 August 2026






Refund eligibility


Eligible customers can request a refund within 
30 calendar days

        of the original purchase.






Processing time


Approved refunds are normally processed within 
7 business days
.






Contact support


Include your order reference when contacting the support team.









    Copyright 2026 Example Support. All rights reserved.
  








The actual HTML extraction contains both knowledge content and website-level content.

We can inspect the source structure before deciding how to clean it.


## 15. Inspect the HTML Structure

In [14]:
print("Title:", soup.title.get_text(strip=True) if soup.title else None)
print("Has <article>:", soup.find("article") is not None)
print("Has <nav>:", soup.find("nav") is not None)
print("Has <footer>:", soup.find("footer") is not None)

Title: Support Knowledge Base — Refunds
Has <article>: True
Has <nav>: True
Has <footer>: True


The source contains semantic structure.

This gives us stronger evidence than searching the extracted text for words such as `"Home"` or `"Copyright"`.


## 16. Isolate the Knowledge-Bearing Article



In [15]:
article = soup.find("article")

if article is None:
    raise ValueError("No <article> element found.")

article_raw = article.get_text("\n", strip=True)

print(article_raw)

Billing
Refunds and Cancellations
Updated 18 August 2026
Refund eligibility
Eligible customers can request a refund within
30 calendar days
of the original purchase.
Processing time
Approved refunds are normally processed within
7 business days
.
Contact support
Include your order reference when contacting the support team.


This removes website navigation and footer content by using the document's actual structure.

That is a document-aware cleaning operation.


## 17. Normalize the Actual Article Text



In [16]:
article_clean = unicodedata.normalize("NFC", article_raw)
article_clean = re.sub(r"[ \t]+", " ", article_clean)
article_clean = re.sub(r"\n{3,}", "\n\n", article_clean)
article_clean = article_clean.strip()

print(article_clean)

Billing
Refunds and Cancellations
Updated 18 August 2026
Refund eligibility
Eligible customers can request a refund within
30 calendar days
of the original purchase.
Processing time
Approved refunds are normally processed within
7 business days
.
Contact support
Include your order reference when contacting the support team.


Here the transformations are applied to the actual article extracted from the real HTML document.


# Part D — What Cleaning Should Not Do

## 18. Do Not Guess Semantic Corrections

Cleaning should not blindly change:

- names;
- IDs;
- dates;
- currency values;
- amounts;
- legal language;
- medical terminology.

If the parser produces ambiguous content, preserve the evidence and use a later stage with the appropriate context.


## 19. Do Not Destroy Structure

Avoid cleaning rules that flatten:

- headings;
- paragraphs;
- lists;
- table columns;
- page boundaries;
- reading order.

For example, replacing every newline with a space might make text look compact while destroying information required for structure preservation and chunking.


## 20. Cleaning vs Structure Preservation

Some problems belong to the next stage.

```text
Cleaning
  ↓
representation-level normalization

Structure Preservation
  ↓
headings
sections
paragraphs
lists
tables
reading order
```

A merged table column is not automatically a whitespace problem.

A repeated header is not automatically safe to delete.

The correct operation depends on evidence from the document.


## 21. Validation

After cleaning, verify that important information survived.


In [17]:
def validate_required_content(text, required_values):
    return {
        value: value in text
        for value in required_values
    }

print("DOCX validation:")
print(validate_required_content(
    docx_clean,
    [
        "OPS-SUP-2026-Q3",
        "Executive Summary",
        "Refund Handling",
        "Document Control",
    ],
))

print("\nHTML validation:")
print(validate_required_content(
    article_clean,
    [
        "Refunds and Cancellations",
        "30 calendar days",
        "7 business days",
        "order reference",
    ],
))

DOCX validation:
{'OPS-SUP-2026-Q3': True, 'Executive Summary': True, 'Refund Handling': True, 'Document Control': True}

HTML validation:
{'Refunds and Cancellations': True, '30 calendar days': True, '7 business days': True, 'order reference': True}


Validation makes the cleaning step testable.

The principle is:

> **A cleaner should make the representation more consistent without silently deleting information.**


## 22. A Conservative Cleaning Function

The generic function below contains only representation-level operations.



In [18]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

This function is intentionally limited.

It does not attempt to solve:

- OCR correction;
- table reconstruction;
- semantic correction;
- heading detection;
- reading-order reconstruction.

Those require different techniques and evidence.


## 23. Exercise

Use the actual documents in `data/`.

For each document:

1. Extract the real parser output.
2. Inspect the representation with `repr()`.
3. Profile actual artifacts.
4. Record which artifacts are present and which are absent.
5. Apply only justified cleaning.
6. Compare before and after.
7. Validate important information.
8. Identify problems that should be handled by structure preservation instead.
9. Explain why a zero artifact count is a valid result.
10. Explain why fabricated Python strings would not demonstrate document-ingestion behavior.


## Key Takeaways

1. **The document is the source of truth.**
2. Inspect actual extraction before cleaning.
3. Do not invent artifacts.
4. A zero count is a valid observation.
5. A cleaning function can legitimately make no change.
6. Demonstrate transformations using real extracted content when the artifact actually exists.
7. Use source/document structure when it provides stronger evidence.
8. Do not guess semantic or table corrections.
9. Validate information after cleaning.
10. Cleaning should reduce representation noise without reducing document information.


## What's Next?

We now have:

```text
Real document
      ↓
Actual extraction
      ↓
Evidence-based cleaning
      ↓
Validated representation
```

Next:

## Structure Preservation

We will preserve the organization of the actual documents:

- titles
- headings
- sections
- subsections
- paragraphs
- lists
- tables
- reading order
